# Serialized Table Baseline vs Cell-Aligned CNN Residual

Choose configuration names as text below. Every checkpoint, history file, validation prediction, and evaluator cache is written directly to Google Drive. Rerun the training cell after a disconnect to resume each selected experiment.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")

if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
%cd /content/table-cnn-mrc

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/cnn_qwen_table_mcr/outputs")
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"All run files will be saved directly under: {DRIVE_OUTPUT_ROOT}")

## Choose configurations

The notebook runs all six configurations by default. To run only a subset, replace `all` with one configuration name per line.

In [ ]:
CONFIGURATIONS_TO_RUN = "all"

AVAILABLE_CONFIGURATIONS = [
    "serialized_table_lora",
    "cnn_residual_mean_middle",
    "cnn_residual_attention_middle",
    "cnn_residual_deeper_middle",
    "cnn_residual_mean_early",
    "cnn_residual_mean_late",
]

requested = [name.strip() for name in CONFIGURATIONS_TO_RUN.splitlines() if name.strip()]
selected_configurations = AVAILABLE_CONFIGURATIONS if requested == ["all"] else requested
unknown = sorted(set(selected_configurations) - set(AVAILABLE_CONFIGURATIONS))
if unknown:
    raise ValueError(f"Unknown configuration names: {unknown}")
if not selected_configurations:
    raise ValueError("Select at least one configuration")

print("Selected configurations:")
for position, name in enumerate(selected_configurations, start=1):
    print(f"  {position}. {name}")

## Optional smoke test

This loads the first selected configuration and checks token-to-cell alignment, loss, gradients, and generation before a long run.

In [ ]:
SMOKE_TEST_CONFIGURATION = next(
    (name for name in selected_configurations if name.startswith("cnn_residual")),
    selected_configurations[0],
)
subprocess.run(
    [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts/smoke_test.py"),
        "--config",
        str(REPO_DIR / "configs" / f"{SMOKE_TEST_CONFIGURATION}.yaml"),
    ],
    cwd=REPO_DIR,
    check=True,
)

## Train or resume selected configurations

Outputs are primary Google Drive directories, not local mirrors. A completed configuration exits immediately; an interrupted configuration resumes from `checkpoint_last.pt`.

In [ ]:
import shlex

for position, name in enumerate(selected_configurations, start=1):
    config_path = REPO_DIR / "configs" / f"{name}.yaml"
    drive_output = DRIVE_OUTPUT_ROOT / name
    drive_output.mkdir(parents=True, exist_ok=True)
    separator = "#" * 88
    print(f"\n{separator}", flush=True)
    print(f"CONFIGURATION {position}/{len(selected_configurations)}: {name}", flush=True)
    print(f"DIRECT DRIVE OUTPUT: {drive_output}", flush=True)
    print(f"{separator}\n", flush=True)
    command = [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts/run_experiment.py"),
        "--config",
        str(config_path),
        "--output-dir",
        str(drive_output),
    ]
    command_text = " ".join(shlex.quote(str(part)) for part in command)
    return_code = get_ipython().system(
        f"cd {shlex.quote(str(REPO_DIR))} && PYTHONUNBUFFERED=1 {command_text}"
    )
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Compare available results

In [ ]:
import json
import pandas as pd
from IPython.display import display

rows = []
for name in AVAILABLE_CONFIGURATIONS:
    history_path = DRIVE_OUTPUT_ROOT / name / "history.json"
    if not history_path.is_file():
        continue
    with history_path.open(encoding="utf-8") as handle:
        history = json.load(handle)
    epochs = history.get("epochs", [])
    rows.append({
        "configuration": name,
        "status": history.get("status"),
        "epochs completed": len(epochs),
        "primary metric": history.get("primary_metric"),
        "best validation score": history.get("best_metric"),
        "latest training loss": epochs[-1].get("training_loss") if epochs else None,
    })
results = pd.DataFrame(rows)
if not results.empty:
    results = results.sort_values("best validation score", ascending=False)
display(results)